In [25]:
#import and config
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import faiss

processed_path = "../dataset/processed"
models_path = "../Models"

movies = pd.read_parquet(
    os.path.join(processed_path, "movie_features.parquet")
)

user_features = pd.read_parquet(
    os.path.join(processed_path, "user_features.parquet")
)

print("Movies:", movies.shape)
print("Users:", user_features.shape)
print("TensorFlow:", tf.__version__)
print("FAISS:", faiss.__version__)

Movies: (3883, 20)
Users: (6040, 4)
TensorFlow: 2.21.0
FAISS: 1.16.0


In [26]:
#loading the FAISS index and movie IDs
movie_index = faiss.read_index(
    os.path.join(models_path, "movie_faiss.index")
)

movie_ids = np.load(
    os.path.join(models_path, "movie_ids.npy")
)

print("FAISS index size:", movie_index.ntotal)
print("Movie IDs:", movie_ids.shape)

FAISS index size: 3456
Movie IDs: (3456,)


In [27]:
# Model Architecture Definitions

import tensorflow as tf
import tf_keras as keras

embedding_dim = 64

class UserModel(keras.Model):
    def __init__(self, user_ids):
        super().__init__()

        self.user_lookup = keras.layers.StringLookup(
            vocabulary=user_ids,
            mask_token=None
        )

        self.user_embedding = keras.layers.Embedding(
            input_dim=len(user_ids) + 1,
            output_dim=embedding_dim
        )

    def call(self, user_id):
        user_index = self.user_lookup(user_id)
        return self.user_embedding(user_index)


class MovieModel(keras.Model):
    def __init__(self, movie_ids):
        super().__init__()

        self.movie_lookup = keras.layers.StringLookup(
            vocabulary=movie_ids,
            mask_token=None
        )

        self.movie_embedding = keras.layers.Embedding(
            input_dim=len(movie_ids) + 1,
            output_dim=embedding_dim
        )

    def call(self, movie_id):
        movie_index = self.movie_lookup(movie_id)
        return self.movie_embedding(movie_index)


print("Model architectures defined successfully.")

Model architectures defined successfully.


In [28]:
#loading thr two tower trained models
retrieval_interactions = pd.read_parquet(
    os.path.join(processed_path, "retrieval_interactions.parquet")
)

retrieval_sorted = retrieval_interactions.sort_values(
    ["user_id", "timestamp"]
).copy()

retrieval_sorted["position"] = retrieval_sorted.groupby("user_id").cumcount()
retrieval_sorted["user_count"] = retrieval_sorted.groupby("user_id")["user_id"].transform("size")

retrieval_sorted["test_count"] = np.where(
    retrieval_sorted["user_count"] >= 2,
    np.maximum(1, np.ceil(retrieval_sorted["user_count"] * 0.2)),
    0
)

retrieval_train = retrieval_sorted[
    retrieval_sorted["position"] <
    retrieval_sorted["user_count"] - retrieval_sorted["test_count"]
]

user_ids = sorted(retrieval_train["user_id"].astype(str).unique())
movie_ids_vocab = sorted(movie_ids.astype(str).tolist())

user_model = UserModel(user_ids)
movie_model = MovieModel(movie_ids_vocab)

user_model(tf.constant(["1"]))
movie_model(tf.constant([movie_ids_vocab[0]]))

user_model.load_weights(os.path.join(models_path, "user_model.weights.h5"))
movie_model.load_weights(os.path.join(models_path, "movie_model.weights.h5"))

print("Training users:", len(user_ids))
print("Training movies:", len(movie_ids_vocab))
print("Both trained models loaded successfully.")

Training users: 6038
Training movies: 3456
Both trained models loaded successfully.


In [29]:
#loading thr ranking model 
ranking_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(129,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

ranking_model.load_weights(
    os.path.join(models_path, "ranking_model.weights.h5")
)

print("Ranking model loaded successfully.")

Ranking model loaded successfully.


In [30]:
#preparing the movie title and watched history 
movies_raw = pd.read_csv(
    "../dataset/ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)

movies_raw["movie_id"] = movies_raw["movie_id"].astype(str)

watched_movies = (
    retrieval_train.groupby("user_id")["movie_id"]
    .apply(lambda x: set(x.astype(str)))
    .to_dict()
)

print("Movie titles loaded:", len(movies_raw))
print("Users with watched history:", len(watched_movies))

Movie titles loaded: 3883
Users with watched history: 6038


In [31]:
#triton and redis recommendation pipline 
import requests
import redis

redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

def recommend_movies(user_id, top_k=10, candidate_k=200):
    user_id = str(user_id)

    if user_id not in user_ids:
        raise ValueError("Unknown user ID.")

    cache_key = f"recommendations:triton:{user_id}:{top_k}"

    cached_data = redis_client.get(cache_key)

    if cached_data:
        return pd.DataFrame(json.loads(cached_data))

    user_embedding = user_model(
        tf.constant([user_id])
    ).numpy().astype(np.float32)

    _, indices = movie_index.search(user_embedding, candidate_k)

    candidates = [
        str(movie_ids[idx])
        for idx in indices[0]
        if idx >= 0
    ]

    watched = watched_movies.get(user_id, set())

    candidates = [
        movie_id for movie_id in candidates
        if movie_id not in watched
    ]

    if not candidates:
        return pd.DataFrame()

    candidate_embeddings = movie_model(
        tf.constant(candidates)
    ).numpy().astype(np.float32)

    user_vector = np.repeat(
        user_embedding, len(candidates), axis=0
    )

    similarities = np.sum(
        user_vector * candidate_embeddings,
        axis=1,
        keepdims=True
    )

    features = np.concatenate(
        [user_vector, candidate_embeddings, similarities],
        axis=1
    ).astype(np.float32)

    payload = {
        "inputs": [{
            "name": "input",
            "shape": list(features.shape),
            "datatype": "FP32",
            "data": features.tolist()
        }],
        "outputs": [{"name": "Identity:0"}]
    }

    response = requests.post(
        "http://localhost:8000/v2/models/ranking_model/infer",
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    scores = np.array(
        response.json()["outputs"][0]["data"]
    )

    results = pd.DataFrame({
        "movie_id": candidates,
        "ranking_score": scores
    })

    results = results.merge(
        movies_raw,
        on="movie_id",
        how="left"
    )

    results = (
        results.sort_values("ranking_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    redis_client.set(
        cache_key,
        results.to_json(orient="records"),
        ex=3600
    )

    return results


print("Triton and Redis recommendation function created.")

Triton and Redis recommendation function created.


In [32]:
#Testing the Triton Recommendation Pipeline
recommendations = recommend_movies("1", top_k=10)

print("Recommendations for User 1:")
display(recommendations)

Recommendations for User 1:


,movie_id,ranking_score,title,genres
0,318,0.936538,"Shawshank Redemption, The (1994)",Drama
1,1,0.886286,Toy Story (1995),Animation|Children's|Comedy
2,1197,0.884831,"Princess Bride, The (1987)",Action|Adventure|Comedy|Romance
3,1225,0.882987,Amadeus (1984),Drama
4,1704,0.881611,Good Will Hunting (1997),Drama
5,953,0.873897,It's a Wonderful Life (1946),Drama
6,914,0.867402,My Fair Lady (1964),Musical|Romance
7,912,0.862800,Casablanca (1942),Drama|Romance|War
8,356,0.860127,Forrest Gump (1994),Comedy|Romance|War
9,2324,0.853683,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama


In [33]:
#saving the sample recommendations
import json
sample_recommendations = recommend_movies("1", top_k=10)

sample_recommendations.to_csv(
    "../Outputs/sample_recommendations.csv",
    index=False
)

print("Sample recommendations saved successfully.")

Sample recommendations saved successfully.


In [34]:
#dynamic user recommendation demo interface 
user_id = input("Enter User ID: ")

recommendations = recommend_movies(user_id, top_k=10)

print(f"Top 10 recommendations for User {user_id}:")
display(recommendations)

Enter User ID:  200


Top 10 recommendations for User 200:


,movie_id,ranking_score,title,genres
0,3753,0.783207,"Patriot, The (2000)",Action|Drama|War
1,2355,0.759659,"Bug's Life, A (1998)",Animation|Children's|Comedy
2,1907,0.740965,Mulan (1998),Animation|Children's
3,3793,0.740898,X-Men (2000),Action|Sci-Fi
4,588,0.730536,Aladdin (1992),Animation|Children's|Comedy|Musical
5,2706,0.680967,American Pie (1999),Comedy
6,2394,0.665393,"Prince of Egypt, The (1998)",Animation|Musical
7,2005,0.663444,"Goonies, The (1985)",Adventure|Children's|Fantasy
8,2687,0.658456,Tarzan (1999),Animation|Children's
9,3623,0.645032,Mission: Impossible 2 (2000),Action|Thriller


In [35]:
# Verify recommendation cache
import json

cache_key = f"recommendations:triton:{user_id}:10"
cached_data = redis_client.get(cache_key)

if cached_data:
    cached_recommendations = pd.DataFrame(json.loads(cached_data))
    print(f"Cached recommendations for User {user_id}:")
    display(cached_recommendations)
else:
    print("No cached recommendations found.")

Cached recommendations for User 200:


,movie_id,ranking_score,title,genres
0,3753,0.783207,"Patriot, The (2000)",Action|Drama|War
1,2355,0.759659,"Bug's Life, A (1998)",Animation|Children's|Comedy
2,1907,0.740965,Mulan (1998),Animation|Children's
3,3793,0.740898,X-Men (2000),Action|Sci-Fi
4,588,0.730536,Aladdin (1992),Animation|Children's|Comedy|Musical
5,2706,0.680967,American Pie (1999),Comedy
6,2394,0.665393,"Prince of Egypt, The (1998)",Animation|Musical
7,2005,0.663444,"Goonies, The (1985)",Adventure|Children's|Fantasy
8,2687,0.658456,Tarzan (1999),Animation|Children's
9,3623,0.645032,Mission: Impossible 2 (2000),Action|Thriller


In [36]:
#Redis Cache Hit & Response Time
import time

start_time = time.time()

recommendations = recommend_movies(user_id, top_k=10)

elapsed_time = time.time() - start_time

print(f"User ID: {user_id}")
print("Response time:", round(elapsed_time, 4), "seconds")
display(recommendations)

User ID: 200
Response time: 0.0021 seconds


,movie_id,ranking_score,title,genres
0,3753,0.783207,"Patriot, The (2000)",Action|Drama|War
1,2355,0.759659,"Bug's Life, A (1998)",Animation|Children's|Comedy
2,1907,0.740965,Mulan (1998),Animation|Children's
3,3793,0.740898,X-Men (2000),Action|Sci-Fi
4,588,0.730536,Aladdin (1992),Animation|Children's|Comedy|Musical
5,2706,0.680967,American Pie (1999),Comedy
6,2394,0.665393,"Prince of Egypt, The (1998)",Animation|Musical
7,2005,0.663444,"Goonies, The (1985)",Adventure|Children's|Fantasy
8,2687,0.658456,Tarzan (1999),Animation|Children's
9,3623,0.645032,Mission: Impossible 2 (2000),Action|Thriller
